# 01 – Data exploration: NigBench / AfriHealth-QA

**Goal of this notebook:** understand the dataset before building anything.

We answer these questions:
1. How many rows, unique questions, and answers per question are there? Do they match the dataset card?
2. Are there missing or empty values?
3. How are categories, difficulty levels and languages distributed?
4. Where is the **English** version of each scenario?
5. How long are scenarios, questions and answers?
6. Is there a theme (e.g. maternal/child health, nutrition) with ≥ ~2,000 pairs?

**Dataset:** [intronhealth/nig-bench](https://huggingface.co/datasets/intronhealth/nig-bench), Intron Health, CC BY-NC-SA 4.0.
We only *read* the data here. Nothing is written to the repo except figures in `results/figures/`.

**Before you run this:** accept the dataset terms on Hugging Face and create a `.env` file with `HF_TOKEN=...` (see `.env.example`).

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the project root importable so we can use src/load_data.py
PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
from src.load_data import load_text_data

# Fixed seed so any sampling (e.g. example rows) is the same every run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

## 1. Load the text columns
The first run downloads the parquet files (~881 MB, mostly audio/images) into the Hugging Face cache outside the repo. We keep only the text columns.

In [ ]:
df = load_text_data()
print("Shape (rows, columns):", df.shape)
df.head(3)

In [ ]:
# Column types and one full example row, so we can see what each field looks like
print(df.dtypes)
df.iloc[0].to_dict()

## 2. Counts: do they match the dataset card?
The card claims **~9,303 answers to 7,289 questions**, with **1,435 questions having more than one answer (up to 12)**.
(The Hub metadata separately reports 9,288 rows. Below are the real numbers.)

In [ ]:
n_rows = len(df)
n_questions = df["question_id"].nunique()
n_answers = df["answer_id"].nunique()

answers_per_q = df.groupby("question_id")["answer_id"].nunique()

print(f"Rows                          : {n_rows}")
print(f"Unique answer_id              : {n_answers}")
print(f"Unique question_id            : {n_questions}")
print(f"Questions with >1 answer      : {(answers_per_q > 1).sum()}")
print(f"Max answers for one question  : {answers_per_q.max()}")
print(f"Rows with a repeated answer_id: {df['answer_id'].duplicated().sum()}")
print()
print("Card says: 9,303 answers | 7,289 questions | 1,435 multi-answer questions | max 12")

In [ ]:
# Distribution: how many questions have 1, 2, 3, ... answers
answers_per_q.value_counts().sort_index().rename("num_questions").to_frame()

In [ ]:
# Does every row of the same question share the same scenario/question text?
# (If not, 'question_id' may not mean what we think.)
for col in ["scenario", "question", "category", "difficulty"]:
    n_variants = df.groupby("question_id")[col].nunique()
    print(f"{col:<11}: questions with >1 distinct value = {(n_variants > 1).sum()}")

## 3. Missing and empty values
`NaN` and empty/whitespace-only strings are both counted as missing.

In [ ]:
def is_missing(series):
    # True if the value is NaN/None or a string that is empty after stripping spaces
    return series.isna() | series.astype(str).str.strip().eq("")

missing = pd.DataFrame({
    "missing_count": {c: is_missing(df[c]).sum() for c in df.columns},
})
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing

## 4. Category distribution
The card mentions 57 CHW curriculum topics.

In [ ]:
cat_counts = (
    df.groupby("category")
      .agg(pairs=("answer_id", "count"), questions=("question_id", "nunique"))
      .sort_values("pairs", ascending=False)
)
print("Number of categories:", len(cat_counts))
cat_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(6, 0.28 * len(cat_counts))))
cat_counts["pairs"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("Number of (question, answer) pairs")
ax.set_ylabel("")
ax.set_title("NigBench: pairs per category")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "category_distribution.png"), dpi=150)
plt.show()

### 4b. Could we narrow to one theme?
A rough check using keywords in the **category names**. This is only a helper for deciding:
read the full list above and adjust the keywords, since a keyword can miss or wrongly include a category.

In [ ]:
THEMES = {
    "maternal_child": ["matern", "pregnan", "antenatal", "postnatal", "delivery", "labour", "labor",
                       "newborn", "neonat", "child", "infant", "immuni", "famil"],
    "nutrition": ["nutri", "feeding", "breast", "malnut", "diet"],
}

for theme, keywords in THEMES.items():
    pattern = "|".join(keywords)
    matched = cat_counts[cat_counts.index.str.contains(pattern, case=False, regex=True)]
    print(f"=== {theme}: {len(matched)} categories, {matched['pairs'].sum()} pairs, "
          f"{matched['questions'].sum()} questions")
    print(matched.to_string(), end="\n\n")

## 5. Difficulty, language and project counts

In [ ]:
for col in ["difficulty", "language", "src_lang", "target_lang", "project_name"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string(), end="\n\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["difficulty"].value_counts(dropna=False).plot.bar(ax=axes[0], color="#55A868")
axes[0].set_title("Difficulty")
df["language"].value_counts(dropna=False).plot.bar(ax=axes[1], color="#C44E52")
axes[1].set_title("Language")
for ax in axes:
    ax.set_ylabel("rows")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "difficulty_language.png"), dpi=150)
plt.show()

## 6. Where is the English scenario?
The brief says scenarios may be in Yoruba/Hausa/Pidgin/Igbo with an English translation elsewhere.
We check how `scenario`, `prompt`, `src_text` and `target_text` relate to each other, per language.

In [ ]:
def same_text(a, b):
    # Compare two text columns after normalising whitespace and case
    norm = lambda s: s.fillna("").astype(str).str.split().str.join(" ").str.lower()
    return norm(a) == norm(b)

checks = pd.DataFrame({
    "language": df["language"],
    "scenario==src_text": same_text(df["scenario"], df["src_text"]),
    "scenario==target_text": same_text(df["scenario"], df["target_text"]),
    "scenario==prompt": same_text(df["scenario"], df["prompt"]),
    "question==src_text": same_text(df["question"], df["src_text"]),
    "question==target_text": same_text(df["question"], df["target_text"]),
    "src_text_present": ~is_missing(df["src_text"]),
    "target_text_present": ~is_missing(df["target_text"]),
})
# Share of rows (0–1) where each check is True, per language
checks.groupby("language").mean().round(3)

In [ ]:
# Language pairs used in the translation columns
df.groupby(["language", "src_lang", "target_lang"], dropna=False).size().rename("rows").to_frame()

In [ ]:
# Look at real examples: 2 random rows per language
cols = ["language", "scenario", "question", "prompt", "src_lang", "src_text", "target_lang", "target_text"]
# Shuffle with a fixed seed, then take the first 2 rows of each language
df.sample(frac=1, random_state=SEED).groupby("language", dropna=False).head(2)[cols]

**Notes after running (fill in):**
- Which column holds the English scenario for each language?
- Is `prompt` the same as `scenario`, or something else?
- Are `answer` texts always English?

## 7. Text length statistics (in words)

In [ ]:
lengths = pd.DataFrame({
    col: df[col].fillna("").astype(str).str.split().str.len()
    for col in ["scenario", "question", "answer"]
})
lengths.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, lengths.columns):
    # Clip at the 99th percentile so a few very long texts don't squash the plot
    upper = lengths[col].quantile(0.99)
    lengths[col].clip(upper=upper).plot.hist(ax=ax, bins=40, color="#8172B2")
    ax.set_title(f"{col} length (words, clipped at p99={upper:.0f})")
    ax.set_xlabel("words")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "text_lengths.png"), dpi=150)
plt.show()

In [ ]:
# Answer length by difficulty: are Hard questions answered at greater length?
lengths.assign(difficulty=df["difficulty"]).groupby("difficulty")["answer"].describe().round(1)

## 8. Duplicates

In [ ]:
print("Exact duplicate rows (scenario, question, answer):",
      df.duplicated(subset=["scenario", "question", "answer"]).sum())
print("Duplicate answer texts (any question):", df["answer"].duplicated().sum())
print("Duplicate (scenario, question) across DIFFERENT question_ids:",
      df.drop_duplicates("question_id").duplicated(subset=["scenario", "question"]).sum())

In [ ]:
# The most frequently repeated answer texts (very short/generic answers are a quality concern)
df["answer"].value_counts().head(10)

## 9. Summary (fill in after running)
| Item | Value |
|---|---|
| Rows | |
| Unique questions | |
| Questions with >1 answer | |
| Categories | |
| English scenario column | |
| Largest candidate theme (pairs) | |

**Decision needed:** use all categories, or narrow to one theme with ≥ ~2,000 pairs?